# 02 — District characterisation

What actually differs between Kona and Ka'u, what doesn't, and what cannot be tested at all.

This notebook exists to answer one reviewer question — *why do you pool the two designated origins?* — with measurements rather than assertion. It is deliberately organised around nulls, because most of the axes people assume separate these districts do not.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd

f = pd.read_pickle('data/farm_features.pkl')
MONTHS = [f'month_{i:02d}' for i in range(1, 13)]
R = pd.read_csv('../data/rainfall_monthly_climatology.csv').drop_duplicates('plot_id').set_index('plot_id')
T = pd.read_csv('../data/temp_monthly_climatology.csv').drop_duplicates('plot_id').set_index('plot_id')
k, q = f[f.region == 'kona'], f[f.region == 'kau']
print(f'kona {len(k)} cells, kau {len(q)} cells')

def cohen_d(a, b):
    sp = np.sqrt(((len(a)-1)*a.var() + (len(b)-1)*b.var()) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / sp if sp > 0 else 0.0

def overlap(a, b):
    return max(0.0, min(a.max(), b.max()) - max(a.min(), b.min()))

kona 409 cells, kau 62 cells


## Temperature — no difference

Both level and seasonal amplitude. This is expected: temperature on a tropical island tracks elevation and sun angle, neither of which respects district boundaries.

In [2]:
rows = []
for col, lab in [('temp_mean_annual', 'mean annual temp (C)'),
                 ('temp_range_annual', 'seasonal range (C)')]:
    rows.append([lab, k[col].mean(), q[col].mean(), q[col].mean()-k[col].mean(),
                 cohen_d(q[col], k[col]), overlap(k[col], q[col])])
print(pd.DataFrame(rows, columns=['variable','kona','kau','diff','cohen_d','overlap'])
        .round(3).to_string(index=False))
print('\nSeasonal amplitude is ~3.2 C in both districts.')
print('For scale: projected warming to 2045 is +1.35 C, i.e. 42% of the entire annual cycle.')

            variable   kona    kau   diff  cohen_d  overlap
mean annual temp (C) 21.007 20.811 -0.196   -0.222    2.760
  seasonal range (C)  3.231  3.240  0.008    0.221    0.114

Seasonal amplitude is ~3.2 C in both districts.
For scale: projected warming to 2045 is +1.35 C, i.e. 42% of the entire annual cycle.


## Annual rainfall — a real difference, with overlap

Ka'u is the wetter district. Note this is the *opposite* of what the previous pipeline recorded; see notebook 01. Two independent products agree on the direction (Atlas ratio 1.35, HCDP monthly sums 1.26).

In [3]:
print(f"kona {k.precip_annual.min():.0f}-{k.precip_annual.max():.0f} mm  mean {k.precip_annual.mean():.0f}")
print(f"kau  {q.precip_annual.min():.0f}-{q.precip_annual.max():.0f} mm  mean {q.precip_annual.mean():.0f}")
print(f"Cohen's d (kau - kona) = {cohen_d(q.precip_annual, k.precip_annual):+.2f}")
print(f"overlap = {overlap(k.precip_annual, q.precip_annual):.0f} mm  <- the ranges DO overlap")

kona 862-1667 mm  mean 1321
kau  1193-2228 mm  mean 1783
Cohen's d (kau - kona) = +2.42
overlap = 475 mm  <- the ranges DO overlap


## Rainfall timing — the one axis that separates them

The monthly cycles are anti-correlated. Kona peaks in September, Ka'u in March. Amplitude is near-identical; the whole difference is phase.

`warm_wet_coupling` turns that into one number per cell: the temperature at which the rain arrives, minus the cell's annual mean. Kona's rain falls in its warm season, Ka'u's in its cool season.

In [4]:
mk = R.loc[k.plot_id, MONTHS].values.mean(0)
mq = R.loc[q.plot_id, MONTHS].values.mean(0)
tk = T.loc[k.plot_id, MONTHS].values.mean(0)
tq = T.loc[q.plot_id, MONTHS].values.mean(0)
NAMES = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
print('monthly rainfall (mm)')
print('      ' + ' '.join(f'{n:>5}' for n in NAMES))
print('kona  ' + ' '.join(f'{v:5.0f}' for v in mk))
print('kau   ' + ' '.join(f'{v:5.0f}' for v in mq))
print(f"\ncorrelation between the two monthly cycles: r = {np.corrcoef(mk, mq)[0,1]:+.2f}")
print(f"corr(temp, rain)  kona {np.corrcoef(tk, mk)[0,1]:+.2f}   kau {np.corrcoef(tq, mq)[0,1]:+.2f}")
print(f"\namplitude is NOT the difference:")
print(f"  driest quarter   kona {k.precip_driest_q.mean():5.0f} mm   kau {q.precip_driest_q.mean():5.0f} mm")
print(f"  wettest quarter  kona {k.precip_wettest_q.mean():5.0f} mm   kau {q.precip_wettest_q.mean():5.0f} mm")
print(f"  seasonality      kona {k.precip_seasonality.mean():5.0f}      kau {q.precip_seasonality.mean():5.0f}")

monthly rainfall (mm)
        Jan   Feb   Mar   Apr   May   Jun   Jul   Aug   Sep   Oct   Nov   Dec
kona     62    52    79    83   105   110   120   129   140   105    68    63
kau     148   142   164    84    82    55    80   108   114   135   157   133

correlation between the two monthly cycles: r = -0.58
corr(temp, rain)  kona +0.90   kau -0.46

amplitude is NOT the difference:
  driest quarter   kona   176 mm   kau   216 mm
  wettest quarter  kona   390 mm   kau   458 mm
  seasonality      kona    30      kau    29


In [5]:
ck, cq = k.warm_wet_coupling, q.warm_wet_coupling
print(f"warm/wet coupling (C)")
print(f"  kona  mean {ck.mean():+.3f}   range [{ck.min():+.3f}, {ck.max():+.3f}]")
print(f"  kau   mean {cq.mean():+.3f}   range [{cq.min():+.3f}, {cq.max():+.3f}]")
print(f"  separation gap {ck.min()-cq.max():+.3f} C  -> DISJOINT")
print(f"  Cohen's d {cohen_d(ck, cq):+.1f}")
print(f"\n{(ck > 0).sum()}/{len(ck)} Kona cells positive ({100*(ck>0).mean():.1f}%).")
print("The one exception sits 22 km inland; region is assigned by a bare longitude")
print("cut (SPLIT_LON), so it is more likely mislabelled than anomalous. The")
print("defensible claim is non-overlap, not that every Kona cell is positive.")

warm/wet coupling (C)
  kona  mean +0.302   range [-0.017, +0.385]
  kau   mean -0.153   range [-0.211, -0.074]
  separation gap +0.058 C  -> DISJOINT
  Cohen's d +7.1

408/409 Kona cells positive (99.8%).
The one exception sits 22 km inland; region is assigned by a bare longitude
cut (SPLIT_LON), so it is more likely mislabelled than anomalous. The
defensible claim is non-overlap, not that every Kona cell is positive.


## Soil — differs, but the data is too coarse to trust the magnitude

SSURGO map units are large relative to these districts. Ka'u gets only a handful of distinct values across 62 cells, which shrinks within-district variance and inflates any standardised effect size. Report the direction; do not quote the *d*.

In [6]:
SOIL = ['awc_mean','om_0_30cm','cec_0_30cm','ph_0_30cm','restrictiondepth_cm',
        'drain_ord','sand_0_30cm','clay_0_30cm']
rows = []
for c in SOIL:
    rows.append([c, k[c].nunique(), q[c].nunique(), cohen_d(q[c], k[c]), overlap(k[c], q[c])])
t = pd.DataFrame(rows, columns=['soil var','kona distinct','kau distinct','cohen_d','overlap'])
print(t.round(2).to_string(index=False))
print(f'\nKa\'u has {len(q)} cells but only {q[SOIL].nunique().min()}-{q[SOIL].nunique().max()} distinct values per variable.')
print('At that granularity soil behaves partly as a region label, the same defect')
print('the 4 km rainfall product had. Treat these effect sizes as unreliable.')

           soil var  kona distinct  kau distinct  cohen_d  overlap
           awc_mean             18             5     2.07     0.11
          om_0_30cm             14             6    -1.12    15.75
         cec_0_30cm              9             6    -1.15     8.33
          ph_0_30cm             15             6    -0.72     1.15
restrictiondepth_cm             13             5     0.93    56.00
          drain_ord              5             2    -0.35     1.00
        sand_0_30cm              7             5    -0.03    50.00
        clay_0_30cm              7             5     0.44    21.67

Ka'u has 62 cells but only 2-6 distinct values per variable.
At that granularity soil behaves partly as a region label, the same defect
the 4 km rainfall product had. Treat these effect sizes as unreliable.


## Why "can a model tell the districts apart" is the wrong question

Kona and Ka'u are geographically disjoint, and every environmental variable is spatially autocorrelated. So any classifier separates them, and no spatially honest cross-validation exists: hold out space and you hold out the class.

Demonstrated below — grouping cells into spatial blocks and checking how many blocks contain both districts.

In [7]:
from sklearn.cluster import KMeans
xy = np.column_stack([f.dist_coast_m.values, f.elev_mean.values])  # any spatial proxy
for nb_ in (6, 12, 24):
    g = KMeans(n_clusters=nb_, random_state=0, n_init=10).fit_predict(
        np.column_stack([f.precip_peak_cos.values*0 + f.dist_coast_m.values, f.elev_mean.values]))
    mixed = sum(len(set(f.region.values[g == i])) > 1 for i in range(nb_))
    print(f'  {nb_:2d} spatial blocks -> {mixed} contain both districts, {nb_-mixed} are single-district')
print('\nSingle-district folds mean ROC-AUC is undefined on most folds. A high')
print('classification score here measures geography, not terroir. Report magnitudes')
print('of specific interpretable variables instead, which is what this notebook does.')

   6 spatial blocks -> 0 contain both districts, 6 are single-district
  12 spatial blocks -> 0 contain both districts, 12 are single-district
  24 spatial blocks -> 0 contain both districts, 24 are single-district

Single-district folds mean ROC-AUC is undefined on most folds. A high
classification score here measures geography, not terroir. Report magnitudes
of specific interpretable variables instead, which is what this notebook does.


## Inventory

| axis | verdict | evidence |
|---|---|---|
| temperature level and seasonality | **no difference** | optima 0.15 C apart, range 3.23 vs 3.24 C |
| annual rainfall | Ka'u wetter | d ≈ +2.1, but ranges overlap by 475 mm |
| **rainfall phase / warm-wet coupling** | **disjoint** | r = −0.58 between cycles; coupling ranges do not overlap |
| soil | differs, magnitude unreliable | SSURGO too coarse (5–6 distinct values in Ka'u) |
| terrain / position | circular by construction | no spatially honest holdout exists |
| cup score | **no difference** | +0.02 pts after variety/processing controls (`cup_scores/`) |
| climate trajectory | **no difference** | every contrast CI spans zero at every block size |

One axis separates these districts. Everything else is identical, overlapping, or untestable — which is the empirical justification for pooling them in notebooks 03–05.

## Figure — the anti-phase rainfall regime

The one axis that separates these districts. Temperature (grey, right axis) peaks in August in both; rainfall (coloured, left axis) peaks in September in Kona and March in Ka'u. Kona's rain therefore arrives with the heat and Ka'u's arrives against it.

In [8]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os
os.makedirs('figures', exist_ok=True)
KONA, KAU, F_COL, C_COL = '#c1440e', '#1f6f8b', '#c1440e', '#888888'
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})

fig, axes = plt.subplots(1, 2, figsize=(11, 4.0), sharey=True)
for ax, (nm, mm, col, tcur) in zip(axes, [('Kona', mk, KONA, tk), ("Ka'u", mq, KAU, tq)]):
    ax.bar(range(1, 13), mm, color=col, alpha=.75, width=.7)
    ax.axhline(mm.mean(), color=col, ls=':', lw=1)
    ax.set_xticks(range(1, 13)); ax.set_xticklabels(NAMES, fontsize=8)
    ax.set_title(f'{nm}   (peak {NAMES[int(np.argmax(mm))]})')
    ax.set_xlabel('month')
    a2 = ax.twinx(); a2.plot(range(1, 13), tcur, color='#555555', lw=1.8)
    a2.set_ylim(18.5, 23.5); a2.spines['top'].set_visible(False)
    a2.set_ylabel('mean temperature (°C)' if nm != 'Kona' else '', color='#555555')
    a2.tick_params(axis='y', colors='#555555')
axes[0].set_ylabel('mean monthly rainfall (mm)')
fig.suptitle(f'Anti-phase rainfall on one volcano   (cycles correlate r = {np.corrcoef(mk, mq)[0,1]:+.2f})',
             y=1.02, fontsize=11)
fig.tight_layout(); fig.savefig('figures/02_rainfall_phase.png', dpi=200, bbox_inches='tight')
plt.close(fig); print('figures/02_rainfall_phase.png')

figures/02_rainfall_phase.png


## Figure — what separates the districts, and what doesn't

Left: annual rainfall overlaps substantially. Right: the warm/wet coupling does not overlap at all. Same two districts, same cells — one variable separates them and the other does not.

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
ax = axes[0]
bins = np.linspace(800, 2300, 40)
ax.hist(k.precip_annual, bins=bins, color=KONA, alpha=.65, label='Kona')
ax.hist(q.precip_annual, bins=bins, color=KAU, alpha=.65, label="Ka'u")
ax.set_xlabel('annual rainfall (mm)'); ax.set_ylabel('farm cells')
ax.set_title(f'Annual amount — overlaps by {overlap(k.precip_annual, q.precip_annual):.0f} mm')
ax.legend(frameon=False)

ax = axes[1]
bins = np.linspace(-.30, .45, 45)
ax.hist(k.warm_wet_coupling, bins=bins, color=KONA, alpha=.65, label='Kona')
ax.hist(q.warm_wet_coupling, bins=bins, color=KAU, alpha=.65, label="Ka'u")
ax.axvline(0, color='k', ls=':', lw=1)
ax.axvspan(q.warm_wet_coupling.max(), k.warm_wet_coupling.min(), color='k', alpha=.07)
ax.set_xlabel('rain-weighted minus mean temperature (°C)')
ax.set_title(f'Warm/wet coupling — disjoint, {k.warm_wet_coupling.min()-q.warm_wet_coupling.max():.3f} °C gap')
ax.legend(frameon=False)
fig.tight_layout(); fig.savefig('figures/02_separation.png', dpi=200, bbox_inches='tight')
plt.close(fig); print('figures/02_separation.png')

figures/02_separation.png
